# Data model

The cleaned tables are reshaped here into a star schema: two fact tables holding the things that happened, surrounded by dimension tables holding the things they happened to.

## Why two fact tables

Delivery and revenue live at different grains. An order is late once, regardless of how many items it contains, so lateness belongs to a table with one row per order. Price belongs to a table with one row per item, because each item has its own.

Forcing both into one table breaks one of them. Join orders to items and every order-level value is repeated once per item: an order delivered in 10 days containing 3 items contributes its delivery time three times, and the average delivery time silently becomes an average weighted by basket size. Forcing everything the other way loses product detail, since an order containing five products has no single product.

So: `fct_orders` at order grain, `fct_order_items` at item grain, sharing their dimensions.

## What gets built

| Table | Grain | Purpose |
| --- | --- | --- |
| `fct_orders` | one delivered order | delivery, promise, review, payment |
| `fct_order_items` | one item on a delivered order | revenue, freight, product, seller |
| `dim_date` | one calendar day | every time axis |
| `dim_customer` | one customer record | where the order went |
| `dim_seller` | one seller | where it came from |
| `dim_product` | one product | what was sold |
| `dim_route` | one seller state to customer state pair | the delivery lane |

Four dimensions - `dim_date`, `dim_seller`, `dim_customer` and `dim_route` - are **conformed**: both fact tables carry their keys, so a filter on any of them applies to delivery and revenue at once, and the two halves of the report can never disagree about what "SP sellers in March" means.

Only `dim_product` reaches a single fact, and necessarily so: an order containing five products has no single product. That is a real limitation of this shape rather than an oversight, and it is worth leaving visible rather than hiding behind an invented "main product" column.

## Scope

Delivered orders only, as the project scope requires: 96,470 orders. Cancelled and unavailable orders stay in `clean.orders`, where the delivery analysis uses them to count orders that never arrived, and are excluded here.

In [16]:
import duckdb

con = duckdb.connect("../olist.duckdb")
con.sql("CREATE SCHEMA IF NOT EXISTS model")

def show(sql, n=20):
    con.sql(sql).show(max_rows=n)

def check(label, sql, expected=None):
    got = con.execute(sql).fetchall()[0][0]
    if expected is None:
        print(f"{label}: {got:,}" if isinstance(got, int) else f"{label}: {got}")
    else:
        print(f"{label}: {got:,} (expected {expected:,}) {'OK' if got == expected else 'MISMATCH'}")

print("connected")

connected


---

## 1. dim_date

**Grain:** one calendar day.

A timestamp such as `2017-10-02 11:07:15` is a moment, not something to group by. Charting orders per month requires "October 2017" to exist as a value, with a defined sort position.

Deriving it on the fly instead causes three problems.

**Days with no activity disappear.** The data covers September 2016 to October 2018, but November 2016 contains no orders at all. An axis built from the dates present in the data jumps from October to December as though November never existed - the gap vanishes instead of showing as zero, which matters precisely when analysing volume.

**Month names sort alphabetically.** Extracted as text they order as April, August, December. A date table carries `month_number` beside `month_name` so one can sort the other.

**Time intelligence requires it.** DAX functions such as `TOTALYTD` and `SAMEPERIODLASTYEAR` need a table marked as the date table, holding one unbroken row per day.

The table spans the earliest purchase to the latest of any date in the data, so that delivery dates and promised dates also resolve.

In [17]:
show("""
    SELECT
        MIN(CAST(purchase_ts AS DATE))  AS first_purchase,
        MAX(CAST(purchase_ts AS DATE))  AS last_purchase,
        MAX(delivered_date)             AS last_delivery,
        MAX(promised_date)              AS last_promise
    FROM clean.orders
""")

# months with no orders at all -- the gap a date table makes visible
show("""
    WITH months AS (
        SELECT DISTINCT strftime(purchase_ts, '%Y-%m') AS ym
        FROM clean.orders
    )
    SELECT ym, LEAD(ym) OVER (ORDER BY ym) AS next_month_present
    FROM months
    QUALIFY date_diff('month', strptime(ym, '%Y-%m'),
                      strptime(LEAD(ym) OVER (ORDER BY ym), '%Y-%m')) > 1
""")

┌────────────────┬───────────────┬───────────────┬──────────────┐
│ first_purchase │ last_purchase │ last_delivery │ last_promise │
│      date      │     date      │     date      │     date     │
├────────────────┼───────────────┼───────────────┼──────────────┤
│ 2016-09-04     │ 2018-10-17    │ 2018-10-17    │ 2018-11-12   │
└────────────────┴───────────────┴───────────────┴──────────────┘

┌─────────┬────────────────────┐
│   ym    │ next_month_present │
│ varchar │      varchar       │
├─────────┼────────────────────┤
│ 2016-10 │ 2016-12            │
└─────────┴────────────────────┘



In [18]:
con.sql("""
    CREATE OR REPLACE TABLE model.dim_date AS
    WITH bounds AS (
        SELECT
            MIN(CAST(purchase_ts AS DATE)) AS d0,
            GREATEST(MAX(CAST(purchase_ts AS DATE)),
                     MAX(delivered_date),
                     MAX(promised_date))   AS d1
        FROM clean.orders
    )
    SELECT
        CAST(d AS DATE)                        AS date,
        YEAR(d)                                AS year,
        'Q' || QUARTER(d)                      AS quarter,
        MONTH(d)                               AS month_number,
        MONTHNAME(d)                           AS month_name,
        strftime(d, '%Y-%m')                   AS year_month,
        DAYNAME(d)                             AS day_name,
        DAYNAME(d) IN ('Saturday', 'Sunday')   AS is_weekend
    FROM bounds, generate_series(bounds.d0, bounds.d1, INTERVAL 1 DAY) t(d)
""")

check("dim_date rows",      "SELECT COUNT(*) FROM model.dim_date")
check("one row per day",    "SELECT COUNT(*) - COUNT(DISTINCT date) FROM model.dim_date", 0)
check("no gaps",            """SELECT date_diff('day', MIN(date), MAX(date)) + 1 - COUNT(*) FROM model.dim_date""", 0)
show("SELECT * FROM model.dim_date WHERE date BETWEEN '2016-11-01' AND '2016-11-04' ORDER BY date")

dim_date rows: 800
one row per day: 0 (expected 0) OK
no gaps: 0 (expected 0) OK
┌────────────┬───────┬─────────┬──────────────┬────────────┬────────────┬───────────┬────────────┐
│    date    │ year  │ quarter │ month_number │ month_name │ year_month │ day_name  │ is_weekend │
│    date    │ int64 │ varchar │    int64     │  varchar   │  varchar   │  varchar  │  boolean   │
├────────────┼───────┼─────────┼──────────────┼────────────┼────────────┼───────────┼────────────┤
│ 2016-11-01 │  2016 │ Q4      │           11 │ November   │ 2016-11    │ Tuesday   │ false      │
│ 2016-11-02 │  2016 │ Q4      │           11 │ November   │ 2016-11    │ Wednesday │ false      │
│ 2016-11-03 │  2016 │ Q4      │           11 │ November   │ 2016-11    │ Thursday  │ false      │
│ 2016-11-04 │  2016 │ Q4      │           11 │ November   │ 2016-11    │ Friday    │ false      │
└────────────┴───────┴─────────┴──────────────┴────────────┴────────────┴───────────┴────────────┘



---

## 2. State regions

Brazil has 27 states, which is too many to read on a chart axis and too few to group usefully without a scheme. The country's five official regions are the standard grouping, and the northeast is where the delivery problems concentrate, so this lookup is used by three dimensions.

This is reference data rather than anything derived from the dataset.

In [19]:
con.sql("""
    CREATE OR REPLACE TABLE model.state_region AS
    SELECT * FROM (VALUES
        ('AC','North'),('AP','North'),('AM','North'),('PA','North'),
        ('RO','North'),('RR','North'),('TO','North'),
        ('AL','Northeast'),('BA','Northeast'),('CE','Northeast'),('MA','Northeast'),
        ('PB','Northeast'),('PE','Northeast'),('PI','Northeast'),('RN','Northeast'),
        ('SE','Northeast'),
        ('DF','Central-West'),('GO','Central-West'),('MT','Central-West'),('MS','Central-West'),
        ('ES','Southeast'),('MG','Southeast'),('RJ','Southeast'),('SP','Southeast'),
        ('PR','South'),('RS','South'),('SC','South')
    ) AS t(state, region)
""")

check("states mapped", "SELECT COUNT(*) FROM model.state_region", 27)
check("customer states unmapped",
      """SELECT COUNT(DISTINCT state) FROM clean.customers c
         WHERE NOT EXISTS (SELECT 1 FROM model.state_region r WHERE r.state = c.state)""", 0)
check("seller states unmapped",
      """SELECT COUNT(DISTINCT state) FROM clean.sellers s
         WHERE NOT EXISTS (SELECT 1 FROM model.state_region r WHERE r.state = s.state)""", 0)

states mapped: 27 (expected 27) OK
customer states unmapped: 0 (expected 0) OK
seller states unmapped: 0 (expected 0) OK


---

## 3. dim_customer

**Grain:** one `customer_id`, which is one customer record attached to one order.

This grain is unusual and is Olist's doing: `customer_id` is regenerated for every order, while `customer_unique_id` identifies the person. The fact table carries `customer_id`, so the dimension must match it, with `customer_unique_id` kept as an attribute for repeat-purchase analysis.

The descriptive columns are already resolved in cleaning. All this step adds is the region.

In [20]:
con.sql("""
    CREATE OR REPLACE TABLE model.dim_customer AS
    SELECT
        c.customer_id,
        c.customer_unique_id,
        c.zip,
        c.city,
        c.state,
        r.region,
        c.lat,
        c.lng,
        c.has_coordinates
    FROM clean.customers c
    LEFT JOIN model.state_region r ON r.state = c.state
""")

check("dim_customer rows",   "SELECT COUNT(*) FROM model.dim_customer", 99441)
check("customer_id unique",  "SELECT COUNT(*) - COUNT(DISTINCT customer_id) FROM model.dim_customer", 0)
check("region missing",      "SELECT COUNT(*) FROM model.dim_customer WHERE region IS NULL", 0)
show("SELECT region, COUNT(*) AS customers FROM model.dim_customer GROUP BY 1 ORDER BY 2 DESC")

dim_customer rows: 99,441 (expected 99,441) OK
customer_id unique: 0 (expected 0) OK
region missing: 0 (expected 0) OK
┌──────────────┬───────────┐
│    region    │ customers │
│   varchar    │   int64   │
├──────────────┼───────────┤
│ Southeast    │     68266 │
│ South        │     14148 │
│ Northeast    │      9394 │
│ Central-West │      5782 │
│ North        │      1851 │
└──────────────┴───────────┘



---

## 4. dim_seller

**Grain:** one seller.

Descriptive attributes only. Performance metrics - on-time rate, average dispatch time, review score - are deliberately excluded: they are measurements, they belong in a fact table or a DAX measure, and a dimension that changes whenever new orders arrive stops being a stable list of things.

In [21]:
con.sql("""
    CREATE OR REPLACE TABLE model.dim_seller AS
    SELECT
        s.seller_id,
        s.zip,
        s.city,
        s.state,
        r.region,
        s.lat,
        s.lng,
        s.has_coordinates,
        s.state_mismatch
    FROM clean.sellers s
    LEFT JOIN model.state_region r ON r.state = s.state
""")

check("dim_seller rows",  "SELECT COUNT(*) FROM model.dim_seller", 3095)
check("seller_id unique", "SELECT COUNT(*) - COUNT(DISTINCT seller_id) FROM model.dim_seller", 0)
show("SELECT region, COUNT(*) AS sellers FROM model.dim_seller GROUP BY 1 ORDER BY 2 DESC")

dim_seller rows: 3,095 (expected 3,095) OK
seller_id unique: 0 (expected 0) OK
┌──────────────┬─────────┐
│    region    │ sellers │
│   varchar    │  int64  │
├──────────────┼─────────┤
│ Southeast    │    2266 │
│ South        │     689 │
│ Central-West │      80 │
│ Northeast    │      56 │
│ North        │       4 │
└──────────────┴─────────┘



---

## 5. dim_product

**Grain:** one product.

A straight pass-through of `clean.products`, which already carries the English category and the computed volume. A `size_band` is added, since freight is driven by bulk and a three-way split reads better on a chart than a continuous volume axis.

In [22]:
show("""
    SELECT
        ROUND(QUANTILE_CONT(volume_cm3, 0.33)) AS p33_volume,
        ROUND(QUANTILE_CONT(volume_cm3, 0.66)) AS p66_volume,
        ROUND(MAX(volume_cm3))                 AS max_volume
    FROM clean.products WHERE has_dimensions
""")

┌────────────┬────────────┬────────────┐
│ p33_volume │ p66_volume │ max_volume │
│   double   │   double   │   double   │
├────────────┼────────────┼────────────┤
│     3960.0 │    12500.0 │   296208.0 │
└────────────┴────────────┴────────────┘



In [23]:
con.sql("""
    CREATE OR REPLACE TABLE model.dim_product AS
    SELECT
        product_id,
        category,
        category_pt,
        weight_g,
        length_cm,
        height_cm,
        width_cm,
        volume_cm3,
        CASE
            WHEN volume_cm3 IS NULL   THEN 'unknown'
            WHEN volume_cm3 <  5000   THEN 'small'
            WHEN volume_cm3 < 20000   THEN 'medium'
            ELSE 'large'
        END                              AS size_band,
        photos_qty,
        description_length,
        has_dimensions
    FROM clean.products
""")

check("dim_product rows",  "SELECT COUNT(*) FROM model.dim_product", 32951)
check("product_id unique", "SELECT COUNT(*) - COUNT(DISTINCT product_id) FROM model.dim_product", 0)
show("SELECT size_band, COUNT(*) AS products FROM model.dim_product GROUP BY 1 ORDER BY 2 DESC")

dim_product rows: 32,951 (expected 32,951) OK
product_id unique: 0 (expected 0) OK
┌───────────┬──────────┐
│ size_band │ products │
│  varchar  │  int64   │
├───────────┼──────────┤
│ small     │    13477 │
│ medium    │    11664 │
│ large     │     7808 │
│ unknown   │        2 │
└───────────┴──────────┘



---

## 6. Order-level aggregates

Before the order fact table can be built, three tables have to be collapsed to one row per order.

**Items.** An order's value is the sum of its items, but the seller and category are not sums. For the 1,278 orders involving more than one seller, the order is attributed to the seller holding the largest share of the order's value - `main_seller_id`. This is a judgement call and it is recorded on the fact table as `n_sellers`, so any result that depends on it can be tested against single-seller orders alone.

**Payments.** An order can carry up to 29 payment lines. These collapse to a total, the highest instalment count, and the payment type that settled the largest share.

**Reviews.** Already one row per order after cleaning. The before/after delivery flag is computed here because it needs both tables - the finding from section 9 of the cleaning notebook, now made into a column.

In [24]:
con.sql("""
    CREATE OR REPLACE TABLE model.order_items_rollup AS
    WITH by_seller AS (
        SELECT order_id, seller_id, SUM(price) AS seller_value,
               ROW_NUMBER() OVER (PARTITION BY order_id
                                  ORDER BY SUM(price) DESC, seller_id) AS rn
        FROM clean.order_items GROUP BY 1, 2
    ),
    by_category AS (
        SELECT i.order_id, p.category, SUM(i.price) AS category_value,
               ROW_NUMBER() OVER (PARTITION BY i.order_id
                                  ORDER BY SUM(i.price) DESC, p.category) AS rn
        FROM clean.order_items i
        LEFT JOIN model.dim_product p USING (product_id)
        GROUP BY 1, 2
    )
    SELECT
        i.order_id,
        COUNT(*)                            AS n_items,
        COUNT(DISTINCT i.seller_id)         AS n_sellers,
        COUNT(DISTINCT i.product_id)        AS n_products,
        SUM(i.price)                        AS items_value,
        SUM(i.freight)                      AS freight_total,
        SUM(p.weight_g)                     AS weight_g,
        SUM(p.volume_cm3)                   AS volume_cm3,
        MAX(i.shipping_limit_ts)            AS shipping_limit_ts,
        ANY_VALUE(s.seller_id)              AS main_seller_id,
        ANY_VALUE(c.category)               AS main_category
    FROM clean.order_items i
    LEFT JOIN model.dim_product p USING (product_id)
    JOIN by_seller   s ON s.order_id = i.order_id AND s.rn = 1
    JOIN by_category c ON c.order_id = i.order_id AND c.rn = 1
    GROUP BY i.order_id
""")

con.sql("""
    CREATE OR REPLACE TABLE model.payments_rollup AS
    SELECT
        order_id,
        COUNT(*)                                        AS n_payment_lines,
        SUM(payment_value)                              AS payment_total,
        MAX(installments)                               AS max_installments,
        ARG_MAX(payment_type, payment_value)            AS main_payment_type
    FROM clean.payments
    GROUP BY order_id
""")

check("items rollup: one row per order",
      "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM model.order_items_rollup", 0)
check("items rollup keeps every item",
      "SELECT SUM(n_items) FROM model.order_items_rollup", 112650)
check("item revenue, all orders",
      "SELECT ROUND(SUM(items_value), 2) FROM model.order_items_rollup")
check("payments rollup: one row per order",
      "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM model.payments_rollup", 0)
show("SELECT n_sellers, COUNT(*) AS orders FROM model.order_items_rollup GROUP BY 1 ORDER BY 1")

items rollup: one row per order: 0 (expected 0) OK
items rollup keeps every item: 112,650 (expected 112,650) OK
item revenue, all orders: 13591643.7
payments rollup: one row per order: 0 (expected 0) OK
┌───────────┬────────┐
│ n_sellers │ orders │
│   int64   │ int64  │
├───────────┼────────┤
│         1 │  97388 │
│         2 │   1219 │
│         3 │     54 │
│         4 │      3 │
│         5 │      2 │
└───────────┴────────┘



---

## 7. dim_route

**Grain:** one seller state to customer state pair.

Routes are the unit the delivery analysis works in: a parcel going SP to SP is a different proposition from SP to AM, and seller and customer state together describe that better than either alone. Holding the pair in a dimension means a route can be filtered and labelled once rather than reconstructed in every query.

Built only from the pairs that actually occur in delivered orders, so the dimension contains no rows the facts cannot reach.

In [25]:
con.sql("""
    CREATE OR REPLACE TABLE model.dim_route AS
    WITH pairs AS (
        SELECT DISTINCT
            sel.state AS seller_state,
            cus.state AS customer_state
        FROM clean.orders o
        JOIN model.order_items_rollup i ON i.order_id = o.order_id
        JOIN clean.customers cus        ON cus.customer_id = o.customer_id
        JOIN clean.sellers   sel        ON sel.seller_id = i.main_seller_id
        WHERE o.is_delivered
    )
    SELECT
        p.seller_state || ' > ' || p.customer_state  AS route_key,
        p.seller_state,
        p.customer_state,
        rs.region                                    AS seller_region,
        rc.region                                    AS customer_region,
        (p.seller_state = p.customer_state)          AS same_state,
        (rs.region = rc.region)                      AS same_region
    FROM pairs p
    LEFT JOIN model.state_region rs ON rs.state = p.seller_state
    LEFT JOIN model.state_region rc ON rc.state = p.customer_state
""")

check("dim_route rows",   "SELECT COUNT(*) FROM model.dim_route")
check("route_key unique", "SELECT COUNT(*) - COUNT(DISTINCT route_key) FROM model.dim_route", 0)
show("SELECT * FROM model.dim_route ORDER BY route_key LIMIT 5")

dim_route rows: 406
route_key unique: 0 (expected 0) OK
┌───────────┬──────────────┬────────────────┬───────────────┬─────────────────┬────────────┬─────────────┐
│ route_key │ seller_state │ customer_state │ seller_region │ customer_region │ same_state │ same_region │
│  varchar  │   varchar    │    varchar     │    varchar    │     varchar     │  boolean   │   boolean   │
├───────────┼──────────────┼────────────────┼───────────────┼─────────────────┼────────────┼─────────────┤
│ AM > AL   │ AM           │ AL             │ North         │ Northeast       │ false      │ false       │
│ AM > MA   │ AM           │ MA             │ North         │ Northeast       │ false      │ false       │
│ AM > MG   │ AM           │ MG             │ North         │ Southeast       │ false      │ false       │
│ BA > AC   │ BA           │ AC             │ Northeast     │ North           │ false      │ false       │
│ BA > AL   │ BA           │ AL             │ Northeast     │ Northeast       │ false   

---

## 8. fct_orders

**Grain:** one delivered order.

This is where everything meets. The delivery metrics arrive already computed from cleaning; what is added here is anything requiring more than one table.

**Distance** uses the haversine formula on the seller's and customer's zip centroids - the great-circle distance in kilometres. It is an approximation twice over: a zip prefix covers an area rather than a point, and a straight line is not a road. Both limitations are stated rather than hidden, and the figure is still the best available proxy for how far a parcel travelled.

**The review timing flag** compares the review's creation date with the delivery timestamp. On late orders, 83.6% of reviews were written before the parcel arrived, which is large enough that every review figure in this project has to be read with this column in mind, and the seller scorecard uses only reviews written after delivery.

**Foreign keys** are `customer_id`, `main_seller_id`, `purchase_date` and `route_key`. Each resolves to exactly one dimension row, which section 10 verifies.

In [26]:
con.sql("""
    CREATE OR REPLACE TABLE model.fct_orders AS
    SELECT
        -- keys
        o.order_id,
        o.customer_id,
        i.main_seller_id,
        CAST(o.purchase_ts AS DATE)                         AS purchase_date,
        sel.state || ' > ' || cus.state                     AS route_key,

        -- timestamps
        o.purchase_ts,
        o.approved_ts,
        o.carrier_ts,
        o.delivered_ts,
        o.promised_date,
        o.delivered_date,

        -- delivery metrics
        o.delivery_days,
        date_diff('day', CAST(o.purchase_ts AS DATE), o.promised_date) AS promised_days,
        o.days_vs_promise,
        o.is_late,
        o.approval_days,
        o.dispatch_days,
        o.transit_days,
        (o.carrier_ts > i.shipping_limit_ts)                AS seller_missed_handoff,
        o.timestamp_anomaly,

        -- distance between seller and customer zip centroids, km
        CASE WHEN sel.lat IS NOT NULL AND cus.lat IS NOT NULL THEN
            2 * 6371 * ASIN(SQRT(
                POWER(SIN(RADIANS(cus.lat - sel.lat) / 2), 2) +
                COS(RADIANS(sel.lat)) * COS(RADIANS(cus.lat)) *
                POWER(SIN(RADIANS(cus.lng - sel.lng) / 2), 2)))
        END                                                 AS distance_km,

        -- basket
        i.n_items,
        i.n_sellers,
        i.n_products,
        i.items_value,
        i.freight_total,
        i.freight_total / NULLIF(i.items_value, 0)          AS freight_ratio,
        i.weight_g,
        i.volume_cm3,
        i.main_category,

        -- payment
        p.payment_total,
        p.max_installments,
        p.main_payment_type,
        p.n_payment_lines,

        -- review
        r.review_score,
        r.review_creation_ts,
        (r.order_id IS NOT NULL)                            AS has_review,
        (r.review_creation_ts < o.delivered_ts)             AS review_before_delivery,
        date_diff('day', CAST(o.delivered_ts AS DATE),
                         CAST(r.review_creation_ts AS DATE)) AS review_lag_days

    FROM clean.orders o
    JOIN      model.order_items_rollup i ON i.order_id    = o.order_id
    JOIN      clean.customers          cus ON cus.customer_id = o.customer_id
    LEFT JOIN clean.sellers            sel ON sel.seller_id   = i.main_seller_id
    LEFT JOIN model.payments_rollup    p ON p.order_id    = o.order_id
    LEFT JOIN clean.reviews            r ON r.order_id    = o.order_id
    WHERE o.is_delivered
""")

check("fct_orders rows",   "SELECT COUNT(*) FROM model.fct_orders", 96470)
check("order_id unique",   "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM model.fct_orders", 0)
check("without distance",  "SELECT COUNT(*) FROM model.fct_orders WHERE distance_km IS NULL")
check("without review",    "SELECT COUNT(*) FROM model.fct_orders WHERE NOT has_review")
show("""
    SELECT
        ROUND(100 * AVG(is_late::INT), 2)      AS late_rate_pct,
        ROUND(AVG(delivery_days), 1)           AS avg_delivery_days,
        ROUND(AVG(promised_days), 1)           AS avg_promised_days,
        ROUND(AVG(distance_km))                AS avg_distance_km,
        ROUND(AVG(review_score), 2)            AS avg_review
    FROM model.fct_orders
""")

fct_orders rows: 96,470 (expected 96,470) OK
order_id unique: 0 (expected 0) OK
without distance: 477
without review: 646
┌───────────────┬───────────────────┬───────────────────┬─────────────────┬────────────┐
│ late_rate_pct │ avg_delivery_days │ avg_promised_days │ avg_distance_km │ avg_review │
│    double     │      double       │      double       │     double      │   double   │
├───────────────┼───────────────────┼───────────────────┼─────────────────┼────────────┤
│          6.77 │              12.6 │              24.4 │           600.0 │       4.16 │
└───────────────┴───────────────────┴───────────────────┴─────────────────┴────────────┘



---

## 9. fct_order_items

**Grain:** one item on a delivered order.

Scoped to the same orders as `fct_orders`, so the two facts always describe the same population and their totals can be reconciled against each other.

**Four dimension keys are carried down from the order**: `purchase_date`, `customer_id`, `route_key` and the item's own `seller_id` and `product_id`. Each is a legitimate attribute at item grain - an item belongs to an order, which was bought by one customer on one date and travelled one route.

This matters more than it looks. A dimension that connects to only one of the two fact tables filters only half the report: select a state and the order measures respond while revenue silently does not, with no error to warn anyone. Carrying the keys down makes `dim_customer`, `dim_route`, `dim_date` and `dim_seller` all **conformed** - shared by both facts - so any filter applies consistently across the whole model. Only `dim_product` remains single-fact, and necessarily so: an order containing five products has no single product.

In [27]:
con.sql("""
    CREATE OR REPLACE TABLE model.fct_order_items AS
    SELECT
        -- keys
        i.order_id,
        i.item_number,
        i.product_id,
        i.seller_id,
        o.customer_id,
        CAST(o.purchase_ts AS DATE)     AS purchase_date,
        f.route_key,
        -- measures
        i.price,
        i.freight,
        i.price + i.freight             AS item_total,
        i.free_shipping,
        i.shipping_limit_ts
    FROM clean.order_items i
    JOIN clean.orders o      USING (order_id)
    JOIN model.fct_orders f  USING (order_id)
    WHERE o.is_delivered
""")

check("fct_order_items rows", "SELECT COUNT(*) FROM model.fct_order_items")
check("key unique",
      "SELECT COUNT(*) - COUNT(DISTINCT order_id || '|' || item_number) FROM model.fct_order_items", 0)
check("revenue", "SELECT ROUND(SUM(price), 2) FROM model.fct_order_items")
check("rows with no customer_id", "SELECT COUNT(*) FROM model.fct_order_items WHERE customer_id IS NULL", 0)
check("rows with no route_key",   "SELECT COUNT(*) FROM model.fct_order_items WHERE route_key IS NULL")

fct_order_items rows: 110,189
key unique: 0 (expected 0) OK
revenue: 13220248.93
rows with no customer_id: 0 (expected 0) OK
rows with no route_key: 0


---

## 10. Verification

A star schema is only correct if three things hold: each table sits at the grain claimed for it, every foreign key resolves to exactly one dimension row, and the two fact tables agree with each other.

The last check is the one that matters most. `fct_orders.items_value` was built by summing items per order; `fct_order_items.price` holds those same items individually. If the two totals differ, the aggregation is wrong - and that is precisely the error that would otherwise surface as two dashboard pages quoting different revenue.

In [28]:
print("GRAIN")
for t, key in [("dim_date", "date"), ("dim_customer", "customer_id"), ("dim_seller", "seller_id"),
               ("dim_product", "product_id"), ("dim_route", "route_key"), ("fct_orders", "order_id")]:
    n, d = con.execute(
        f"SELECT COUNT(*), COUNT(*) - COUNT(DISTINCT {key}) FROM model.{t}").fetchall()[0]
    print(f"  model.{t:18}{n:>9,} rows   duplicate {key}: {d}")
n, d = con.execute("""SELECT COUNT(*), COUNT(*) - COUNT(DISTINCT order_id || '|' || item_number)
                      FROM model.fct_order_items""").fetchall()[0]
print(f"  model.fct_order_items {n:>9,} rows   duplicate key: {d}")

print("\nFOREIGN KEYS (rows in the fact with no matching dimension row)")
fks = [
    ("fct_orders", "customer_id",    "dim_customer", "customer_id"),
    ("fct_orders", "main_seller_id", "dim_seller",   "seller_id"),
    ("fct_orders", "purchase_date",  "dim_date",     "date"),
    ("fct_orders", "route_key",      "dim_route",    "route_key"),
    ("fct_order_items", "product_id",    "dim_product",  "product_id"),
    ("fct_order_items", "seller_id",     "dim_seller",   "seller_id"),
    ("fct_order_items", "purchase_date", "dim_date",     "date"),
    ("fct_order_items", "customer_id",   "dim_customer", "customer_id"),
    ("fct_order_items", "route_key",     "dim_route",    "route_key"),
]
for fact, fk, dim, dk in fks:
    orphans = con.execute(f"""
        SELECT COUNT(*) FROM model.{fact} f
        LEFT JOIN model.{dim} d ON d.{dk} = f.{fk}
        WHERE f.{fk} IS NOT NULL AND d.{dk} IS NULL""").fetchall()[0][0]
    print(f"  {fact}.{fk:16} -> {dim:14} {orphans}")

print("\nTHE TWO FACTS AGREE")
a = con.execute("SELECT ROUND(SUM(items_value), 2) FROM model.fct_orders").fetchall()[0][0]
b = con.execute("SELECT ROUND(SUM(price), 2) FROM model.fct_order_items").fetchall()[0][0]
print(f"  fct_orders.items_value     {a:>14,.2f}")
print(f"  fct_order_items.price      {b:>14,.2f}   {'OK' if abs(a - b) < 0.01 else 'MISMATCH'}")

c = con.execute("SELECT SUM(n_items) FROM model.fct_orders").fetchall()[0][0]
d2 = con.execute("SELECT COUNT(*) FROM model.fct_order_items").fetchall()[0][0]
print(f"  fct_orders.n_items total   {c:>14,}")
print(f"  fct_order_items row count  {d2:>14,}   {'OK' if c == d2 else 'MISMATCH'}")

print("\nSCOPE")
print(f"  orders in both facts       {con.execute('SELECT COUNT(DISTINCT order_id) FROM model.fct_order_items').fetchall()[0][0]:>14,}")
print(f"  orders in fct_orders       {con.execute('SELECT COUNT(*) FROM model.fct_orders').fetchall()[0][0]:>14,}")

GRAIN
  model.dim_date                800 rows   duplicate date: 0
  model.dim_customer         99,441 rows   duplicate customer_id: 0
  model.dim_seller            3,095 rows   duplicate seller_id: 0
  model.dim_product          32,951 rows   duplicate product_id: 0
  model.dim_route               406 rows   duplicate route_key: 0
  model.fct_orders           96,470 rows   duplicate order_id: 0
  model.fct_order_items   110,189 rows   duplicate key: 0

FOREIGN KEYS (rows in the fact with no matching dimension row)
  fct_orders.customer_id      -> dim_customer   0
  fct_orders.main_seller_id   -> dim_seller     0
  fct_orders.purchase_date    -> dim_date       0
  fct_orders.route_key        -> dim_route      0
  fct_order_items.product_id       -> dim_product    0
  fct_order_items.seller_id        -> dim_seller     0
  fct_order_items.purchase_date    -> dim_date       0
  fct_order_items.customer_id      -> dim_customer   0
  fct_order_items.route_key        -> dim_route      0

THE

---

## 11. Export for Power BI

The model is written out as CSV, one file per table, for Power BI to import. CSV rather than a live DuckDB connection because it needs no ODBC driver and the files are small enough that nothing is gained by anything cleverer.

In Power BI the relationships are created once, each single-direction from dimension to fact, and `dim_date` is marked as the date table so that time intelligence works.

In [29]:
import os
out = "../outputs/model"
os.makedirs(out, exist_ok=True)

tables = ["dim_date", "dim_customer", "dim_seller", "dim_product", "dim_route",
          "fct_orders", "fct_order_items"]

for t in tables:
    con.sql(f"COPY model.{t} TO '{out}/{t}.csv' (HEADER, DELIMITER ',')")
    size = os.path.getsize(f"{out}/{t}.csv") / 1_000_000
    rows = con.execute(f"SELECT COUNT(*) FROM model.{t}").fetchall()[0][0]
    print(f"  {t:18}{rows:>9,} rows   {size:6.1f} MB")

print("\nwritten to outputs/model/")

  dim_date                800 rows      0.0 MB
  dim_customer         99,441 rows     13.9 MB
  dim_seller            3,095 rows      0.3 MB
  dim_product          32,951 rows      3.6 MB
  dim_route               406 rows      0.0 MB
  fct_orders           96,470 rows     44.1 MB
  fct_order_items     110,189 rows     21.9 MB

written to outputs/model/


In [30]:
con.close()
print("connection closed")

connection closed


---

## Relationships to create in Power BI

| From (dimension) | To (fact) | Key | Cardinality | Direction |
| --- | --- | --- | --- | --- |
| `dim_date` | `fct_orders` | date to purchase_date | one to many | single |
| `dim_date` | `fct_order_items` | date to purchase_date | one to many | single |
| `dim_customer` | `fct_orders` | customer_id | one to many | single |
| `dim_customer` | `fct_order_items` | customer_id | one to many | single |
| `dim_route` | `fct_order_items` | route_key | one to many | single |
| `dim_seller` | `fct_orders` | seller_id to main_seller_id | one to many | single |
| `dim_seller` | `fct_order_items` | seller_id | one to many | single |
| `dim_route` | `fct_orders` | route_key | one to many | single |
| `dim_product` | `fct_order_items` | product_id | one to many | single |

Every relationship is single-direction, from the dimension into the fact. Bidirectional filtering is avoided: with two fact tables sharing dimensions it creates ambiguous filter paths, and the results become dependent on evaluation order rather than on the model.

`dim_date` must be marked as a date table in Power BI, otherwise time intelligence functions return incorrect results without raising an error.

One consequence of this shape remains, and it is unavoidable. `dim_product` connects only to `fct_order_items`, because an order containing five products has no single product. So a product or category filter reaches revenue but not order-level measures such as average delivery days, and a visual combining the two will show the unfiltered total on every row - no error, just a number that does not respond.

Either keep order-level measures out of product-sliced visuals, or write the measure to cross the gap explicitly, which forces the question of what "delivery time for a category" means on a multi-category order. Every other dimension is conformed and filters both facts.

**Next:** `04_delivery_analysis.ipynb`.